# 🐦 BirdCLEF+ 2026 — Metadata EDA

> **대회:** https://www.kaggle.com/competitions/birdclef-2026  
> **데이터:** train.csv, taxonomy.csv (메타데이터만)

---
## 목차
1. 데이터 로드 및 기본 정보
2. 분류군(Class) 분포
3. 종(Species)별 샘플 수 — 클래스 불균형
4. 데이터 출처 (XC vs iNat)
5. 음질 평점(Rating) 분포
6. 울음 유형(Type) 분포
7. 지리적 분포 (위/경도)
8. Secondary Labels 분석
9. 라이선스 분포
10. 핵심 인사이트 요약

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')
PALETTE = sns.color_palette('tab10')

DATA_DIR = './data'
print('✅ 패키지 로드 완료')

---
## 1. 데이터 로드 및 기본 정보

In [ ]:
train = pd.read_csv(f'{DATA_DIR}/train.csv')
taxonomy = pd.read_csv(f'{DATA_DIR}/taxonomy.csv')
sample_sub = pd.read_csv(f'{DATA_DIR}/sample_submission.csv')

print(f'train.csv shape   : {train.shape}')
print(f'taxonomy.csv shape: {taxonomy.shape}')
print(f'sample_submission : {sample_sub.shape}')
print(f'\n총 종 수 (제출 컬럼): {sample_sub.shape[1] - 1}종')
print(f'총 예측 행 수      : {sample_sub.shape[0]}행 (테스트 샘플)')

In [ ]:
train.head(3)

In [ ]:
print('=== train.csv 컬럼별 결측값 ===')
missing = train.isnull().sum()
missing_pct = (missing / len(train) * 100).round(1)
print(pd.DataFrame({'결측 수': missing, '결측률(%)': missing_pct})[missing > 0])

print('\n=== 기본 통계 ===')
print(f'총 학습 샘플 수: {len(train):,}개')
print(f'고유 종 수    : {train["primary_label"].nunique()}종')
print(f'고유 저자 수  : {train["author"].nunique():,}명')
print(f'컬렉션 종류   : {train["collection"].unique()}')

In [ ]:
taxonomy.head(10)

---
## 2. 분류군(Class) 분포

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- 종 수 기준 ---
class_species = taxonomy['class_name'].value_counts()
axes[0].pie(class_species, labels=class_species.index, autopct='%1.1f%%',
            colors=PALETTE, startangle=140, pctdistance=0.75)
axes[0].set_title('분류군별 종 수 (taxonomy.csv)', fontsize=13, fontweight='bold')

# --- 샘플 수 기준 ---
class_samples = train['class_name'].value_counts()
bars = axes[1].barh(class_samples.index[::-1], class_samples.values[::-1], color=PALETTE)
for bar, val in zip(bars, class_samples.values[::-1]):
    axes[1].text(bar.get_width() + 50, bar.get_y() + bar.get_height()/2,
                 f'{val:,}', va='center', fontsize=10)
axes[1].set_title('분류군별 학습 샘플 수 (train.csv)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('샘플 수')

plt.tight_layout()
plt.show()

print('\n=== 분류군별 종 수 & 샘플 수 ===')
summary = taxonomy['class_name'].value_counts().rename('종 수').to_frame()
summary['샘플 수'] = train['class_name'].value_counts()
summary['샘플/종 평균'] = (summary['샘플 수'] / summary['종 수']).round(1)
print(summary)

---
## 3. 종별 샘플 수 — 클래스 불균형

In [ ]:
species_counts = train.groupby(['primary_label', 'class_name', 'common_name']).size().reset_index(name='count')
species_counts = species_counts.sort_values('count', ascending=False)

print(f'샘플 수 기초 통계:')
print(species_counts['count'].describe().round(1))
print(f'\n샘플 수 < 10인 종: {(species_counts["count"] < 10).sum()}종')
print(f'샘플 수 < 5인 종 : {(species_counts["count"] < 5).sum()}종')
print(f'샘플 수 1인 종   : {(species_counts["count"] == 1).sum()}종')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(15, 10))

# --- TOP 30 종 ---
top30 = species_counts.head(30)
colors_top = [PALETTE[['Aves','Amphibia','Insecta','Mammalia','Reptilia'].index(c) % 10]
              for c in top30['class_name']]
axes[0].bar(range(len(top30)), top30['count'], color=colors_top)
axes[0].set_xticks(range(len(top30)))
axes[0].set_xticklabels(top30['common_name'], rotation=45, ha='right', fontsize=8)
axes[0].set_title('샘플 수 TOP 30 종', fontsize=13, fontweight='bold')
axes[0].set_ylabel('샘플 수')

# --- 전체 분포 히스토그램 ---
axes[1].hist(species_counts['count'], bins=50, color='steelblue', edgecolor='white')
axes[1].axvline(species_counts['count'].median(), color='red', linestyle='--',
                label=f'중앙값: {species_counts["count"].median():.0f}')
axes[1].axvline(species_counts['count'].mean(), color='orange', linestyle='--',
                label=f'평균: {species_counts["count"].mean():.0f}')
axes[1].set_title('종별 샘플 수 분포 (전체 234종)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('샘플 수')
axes[1].set_ylabel('종 수')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# 하위 30종 (희귀종)
print('=== 샘플 수 하위 30종 (희귀종 위험) ===')
print(species_counts.tail(30)[['common_name','class_name','count']].to_string(index=False))

---
## 4. 데이터 출처 (XC vs iNat)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 전체 비율
coll = train['collection'].value_counts()
axes[0].pie(coll, labels=coll.index, autopct='%1.1f%%',
            colors=['#4C72B0', '#DD8452'], startangle=90)
axes[0].set_title('전체 샘플 출처 비율', fontsize=13, fontweight='bold')

# 분류군별 출처
coll_class = train.groupby(['class_name', 'collection']).size().unstack(fill_value=0)
coll_class.plot(kind='bar', ax=axes[1], color=['#4C72B0', '#DD8452'],
                rot=30, edgecolor='white')
axes[1].set_title('분류군별 출처 비율', fontsize=13, fontweight='bold')
axes[1].set_xlabel('')
axes[1].set_ylabel('샘플 수')
axes[1].legend(title='컬렉션')

plt.tight_layout()
plt.show()

print(coll)

---
## 5. 음질 평점(Rating) 분포

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 전체 rating 분포
rating_counts = train['rating'].value_counts().sort_index()
axes[0].bar(rating_counts.index.astype(str), rating_counts.values, color='steelblue', edgecolor='white')
axes[0].set_title('전체 음질 평점 분포', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Rating')
axes[0].set_ylabel('샘플 수')

# XC vs iNat rating 비교
xc = train[train['collection']=='XC']['rating']
inat = train[train['collection']=='iNat']['rating']
axes[1].hist(xc[xc > 0], bins=20, alpha=0.6, label=f'XC (n={len(xc):,})', color='#4C72B0')
axes[1].hist(inat[inat > 0], bins=20, alpha=0.6, label=f'iNat (n={len(inat):,})', color='#DD8452')
axes[1].set_title('컬렉션별 평점 분포 (rating=0 제외)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Rating')
axes[1].set_ylabel('샘플 수')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'rating=0 (미평가): {(train["rating"]==0).sum():,}개 ({(train["rating"]==0).mean()*100:.1f}%)')
print(f'rating 4~5 (고품질): {(train["rating"]>=4).sum():,}개 ({(train["rating"]>=4).mean()*100:.1f}%)')

---
## 6. 울음 유형(Type) 분포

In [ ]:
# type은 여러 값이 콤마로 구분되어 있을 수 있음
all_types = []
for t in train['type'].dropna():
    for item in str(t).split(','):
        item = item.strip().lower()
        if item:
            all_types.append(item)

type_counts = pd.Series(Counter(all_types)).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 5))
top_types = type_counts.head(20)
ax.barh(top_types.index[::-1], top_types.values[::-1], color='teal')
ax.set_title('울음 유형(Type) TOP 20', fontsize=13, fontweight='bold')
ax.set_xlabel('등장 횟수')
plt.tight_layout()
plt.show()

print(f'type 결측 비율: {train["type"].isnull().mean()*100:.1f}%')
print(f'\nTOP 10 타입:')
print(type_counts.head(10))

---
## 7. 지리적 분포 (위/경도)

In [ ]:
geo = train.dropna(subset=['latitude', 'longitude'])
print(f'위경도 보유 샘플: {len(geo):,} / {len(train):,}')
print(f'위도 범위: {geo["latitude"].min():.1f} ~ {geo["latitude"].max():.1f}')
print(f'경도 범위: {geo["longitude"].min():.1f} ~ {geo["longitude"].max():.1f}')

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# 분류군별 컬러
class_colors = {'Aves': '#4C72B0', 'Amphibia': '#55A868',
                'Insecta': '#C44E52', 'Mammalia': '#8172B2', 'Reptilia': '#CCB974'}

for cls, grp in geo.groupby('class_name'):
    axes[0].scatter(grp['longitude'], grp['latitude'],
                    alpha=0.3, s=5, label=cls, color=class_colors.get(cls, 'gray'))

# 판타날 영역 표시
from matplotlib.patches import Rectangle
rect = Rectangle((-57.6, -21.6), 1.7, 5.1, linewidth=2,
                 edgecolor='red', facecolor='none', linestyle='--', label='판타날 테스트 영역')
axes[0].add_patch(rect)
axes[0].set_title('학습 데이터 지리적 분포 (분류군별)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('경도')
axes[0].set_ylabel('위도')
axes[0].legend(markerscale=3, fontsize=9)
axes[0].grid(True, alpha=0.3)

# 위도/경도 히스토그램
axes[1].hist2d(geo['longitude'], geo['latitude'], bins=60, cmap='YlOrRd')
axes[1].set_title('녹음 밀도 히트맵', fontsize=12, fontweight='bold')
axes[1].set_xlabel('경도')
axes[1].set_ylabel('위도')
plt.colorbar(axes[1].get_children()[0], ax=axes[1], label='샘플 수')

plt.tight_layout()
plt.show()

In [ ]:
# 판타날 영역 내 학습 데이터 비율
pantanal = geo[
    (geo['latitude'].between(-21.6, -16.5)) &
    (geo['longitude'].between(-57.6, -55.9))
]
print(f'판타날 영역 내 학습 샘플: {len(pantanal):,}개 ({len(pantanal)/len(geo)*100:.1f}%)')
print(f'판타날 외 학습 샘플    : {len(geo)-len(pantanal):,}개 ({(len(geo)-len(pantanal))/len(geo)*100:.1f}%)')
print('\n→ 대부분의 학습 데이터는 전 세계에서 수집된 XC/iNat 데이터이며,')
print('  테스트 데이터는 판타날 현장 녹음임 → 도메인 갭(domain gap) 주의!')

---
## 8. Secondary Labels 분석

In [ ]:
# secondary_labels 파싱
train['has_secondary'] = train['secondary_labels'].notna() & (train['secondary_labels'] != '[]') & (train['secondary_labels'] != '')

print(f'secondary_labels 보유 샘플: {train["has_secondary"].sum():,}개 ({train["has_secondary"].mean()*100:.1f}%)')

# secondary_labels가 있는 샘플 수 분포
def count_labels(s):
    if pd.isna(s) or s in ('[]', ''):
        return 0
    return len(str(s).strip("[]").replace("'", "").split(','))

train['n_secondary'] = train['secondary_labels'].apply(count_labels)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

counts = train['n_secondary'].value_counts().sort_index()
axes[0].bar(counts.index[:10], counts.values[:10], color='teal', edgecolor='white')
axes[0].set_title('secondary_labels 개수 분포', fontsize=12, fontweight='bold')
axes[0].set_xlabel('secondary label 수')
axes[0].set_ylabel('샘플 수')
axes[0].set_xticks(range(10))

# 분류군별 secondary 보유율
sec_rate = train.groupby('class_name')['has_secondary'].mean() * 100
axes[1].barh(sec_rate.index, sec_rate.values, color=list(class_colors.values()))
for i, v in enumerate(sec_rate.values):
    axes[1].text(v + 0.5, i, f'{v:.1f}%', va='center')
axes[1].set_title('분류군별 secondary_labels 보유율', fontsize=12, fontweight='bold')
axes[1].set_xlabel('비율 (%)')

plt.tight_layout()
plt.show()

---
## 9. 라이선스 분포

In [ ]:
license_counts = train['license'].value_counts()

fig, ax = plt.subplots(figsize=(10, 5))
license_counts.head(10).plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('라이선스 종류 분포 (TOP 10)', fontsize=12, fontweight='bold')
ax.set_xlabel('샘플 수')
plt.tight_layout()
plt.show()

# CC 라이선스 상업적 이용 가능 여부
nc_mask = train['license'].str.contains('nc', case=False, na=False)
print(f'비상업(NC) 라이선스 비율: {nc_mask.mean()*100:.1f}% ({nc_mask.sum():,}개)')
print(f'상업 가능 라이선스 비율  : {(~nc_mask).mean()*100:.1f}% ({(~nc_mask).sum():,}개)')

---
## 10. 핵심 인사이트 요약

In [ ]:
print('=' * 60)
print('📊 BirdCLEF+ 2026 메타데이터 EDA 핵심 인사이트')
print('=' * 60)

print(f'''
【데이터 규모】
  · 총 학습 샘플: {len(train):,}개
  · 총 종 수    : {train["primary_label"].nunique()}종 (제출 컬럼 234개)
  · 종당 평균   : {len(train)/train["primary_label"].nunique():.0f}개
  · 종당 중앙값 : {species_counts["count"].median():.0f}개

【분류군 구성】
  · 조류(Aves)  : {(train["class_name"]=="Aves").sum():,}개 ({(train["class_name"]=="Aves").mean()*100:.1f}%)
  · 양서류      : {(train["class_name"]=="Amphibia").sum():,}개
  · 곤충        : {(train["class_name"]=="Insecta").sum():,}개
  · 포유류      : {(train["class_name"]=="Mammalia").sum():,}개
  · 파충류      : {(train["class_name"]=="Reptilia").sum():,}개

【클래스 불균형】
  · MAX: {species_counts["count"].max()}개 ({species_counts.iloc[0]["common_name"]})
  · MIN: {species_counts["count"].min()}개 ({species_counts.iloc[-1]["common_name"]})
  · 샘플 <10종: {(species_counts["count"] < 10).sum()}종
  · 불균형 비율(max/min): {species_counts["count"].max()/species_counts["count"].min():.0f}배

【데이터 출처】
  · XC(Xeno-canto): {(train["collection"]=="XC").sum():,}개 ({(train["collection"]=="XC").mean()*100:.1f}%)
  · iNat(iNaturalist): {(train["collection"]=="iNat").sum():,}개 ({(train["collection"]=="iNat").mean()*100:.1f}%)

【음질 평점】
  · 미평가(0): {(train["rating"]==0).sum():,}개 ({(train["rating"]==0).mean()*100:.1f}%)
  · 고품질(4~5): {(train["rating"]>=4).sum():,}개 ({(train["rating"]>=4).mean()*100:.1f}%)

【도메인 갭 주의】
  · 학습 데이터: 전세계 XC/iNat 녹음 (깨끗한 단종 녹음)
  · 테스트 데이터: 판타날 현장 녹음 (복잡한 다종 사운드스케이프)
  → 강력한 augmentation + 도메인 적응 전략 필요

【모델링 시사점】
  ① 심각한 클래스 불균형 → 가중 손실 함수, 오버샘플링 고려
  ② 희귀종({(species_counts["count"] < 10).sum()}종) 특별 처리 필요
  ③ 미평가 iNat 데이터 품질 필터링 검토
  ④ train_soundscapes(현장 녹음) 적극 활용 → 도메인 갭 완화
  ⑤ GPU 비활성화 → CPU 90분 이내 경량 모델 설계 필수
''')